# Test de hipótesis en Python 🦆

**Objetivos**
1. Entender las **tres colas** (bilateral, izquierda, derecha) y sus regiones de **rechazo** y **no rechazo** (aceptación) de H₀.
2. Aplicar tests con `scipy.stats` y `statsmodels` (t, z de proporciones, χ², ANOVA, Mann-Whitney).
3. Interpretar el **error tipo I (α)**, el **error tipo II (β)** y la **potencia (1 − β)**.

**Ruta del notebook**
| Parte | Contenido |
|---|---|
| 1 | Las 3 colas sobre una sola distribución (H₀, H₁, regiones) |
| 2 | Test t de una muestra: los patitos 🦆 |
| 3 | Errores tipo I y II, potencia y tamaño de muestra |
| 4 | Test z de proporciones |
| 5 | χ², ANOVA y Mann-Whitney |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestPower
from statsmodels.stats.proportion import proportions_ztest

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(1)   # semilla fija: resultados reproducibles

## Parte 1 — Las tres colas sobre una sola distribución

Todo test de hipótesis compara un **estadístico** calculado con los datos contra su distribución **suponiendo que H₀ es verdadera**. Usamos acá una única distribución, la normal estándar N(0, 1), y cambiamos solo la **hipótesis alternativa H₁**:

| Tipo de test | H₀ | H₁ | Región de rechazo | `alternative` en scipy |
|---|---|---|---|---|
| Bilateral (dos colas) | μ = μ₀ | μ ≠ μ₀ | las dos colas, α/2 en cada una | `"two-sided"` |
| Cola izquierda | μ ≥ μ₀ (o μ = μ₀) | μ < μ₀ | cola izquierda, área α | `"less"` |
| Cola derecha | μ ≤ μ₀ (o μ = μ₀) | μ > μ₀ | cola derecha, área α | `"greater"` |

- **Región de rechazo**: valores del estadístico tan extremos que, si H₀ fuera cierta, ocurrirían con probabilidad α. Si el estadístico cae ahí → se **rechaza H₀**.
- **Región de no rechazo** (a veces llamada "de aceptación"): el resto, con probabilidad 1 − α. Si cae ahí → **no hay evidencia suficiente para rechazar H₀** (no se "demuestra" H₀).

In [ ]:
def graficar_cola(ax, cola, alpha=0.05, estadistico=None):
    """Dibuja N(0,1) con regiones de rechazo y no rechazo para una cola.
    cola: 'two-sided' | 'less' | 'greater'."""
    x = np.linspace(-4, 4, 800)
    y = stats.norm.pdf(x)
    ax.plot(x, y, color="black", lw=1.5)

    if cola == "two-sided":
        c = stats.norm.ppf(1 - alpha / 2)
        rechazo = [(x <= -c), (x >= c)]
        no_rech = (x >= -c) & (x <= c)
        criticos = [-c, c]
        h0, h1 = r"$H_0:\ \mu=\mu_0$", r"$H_1:\ \mu\neq\mu_0$"
        lab_alpha = [r"$\alpha/2$", r"$\alpha/2$"]
        pos_alpha = [-c - 0.7, c + 0.7]
        titulo = "Bilateral (dos colas)"
    elif cola == "less":
        c = stats.norm.ppf(alpha)
        rechazo = [(x <= c)]
        no_rech = (x >= c)
        criticos = [c]
        h0, h1 = r"$H_0:\ \mu\geq\mu_0$", r"$H_1:\ \mu<\mu_0$"
        lab_alpha = [r"$\alpha$"]
        pos_alpha = [c - 0.7]
        titulo = "Cola izquierda"
    else:
        c = stats.norm.ppf(1 - alpha)
        rechazo = [(x >= c)]
        no_rech = (x <= c)
        criticos = [c]
        h0, h1 = r"$H_0:\ \mu\leq\mu_0$", r"$H_1:\ \mu>\mu_0$"
        lab_alpha = [r"$\alpha$"]
        pos_alpha = [c + 0.7]
        titulo = "Cola derecha"

    ax.fill_between(x, y, where=no_rech, color="tab:green", alpha=0.25,
                    label=r"No rechazo de $H_0$ ($1-\alpha$)")
    for i, m in enumerate(rechazo):
        ax.fill_between(x, y, where=m, color="tab:red", alpha=0.55,
                        label=r"Rechazo de $H_0$" if i == 0 else None)
    for cr in criticos:
        ax.axvline(cr, color="tab:red", ls="--", lw=1)
        ax.text(cr, -0.035, f"{cr:.2f}", ha="center", va="top", color="tab:red", fontsize=9)
    for t, px in zip(lab_alpha, pos_alpha):
        ax.text(px, 0.045, t, ha="center", fontsize=12, color="tab:red")
    if estadistico is not None:
        ax.axvline(estadistico, color="tab:blue", lw=2)
        ax.text(estadistico, 0.41, f"z obs = {estadistico:.2f}", ha="center", color="tab:blue", fontsize=9)

    ax.text(0, 0.15, "NO RECHAZO\n(aceptación de $H_0$)", ha="center", fontsize=9, color="darkgreen")
    ax.set_title(f"{titulo}\n{h0}   {h1}", fontsize=11)
    ax.set_ylim(-0.07, 0.46)
    ax.set_yticks([])
    ax.set_xlabel("Estadístico z bajo $H_0$")


fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), sharey=True)
for ax, cola in zip(axes, ["two-sided", "less", "greater"]):
    graficar_cola(ax, cola, alpha=0.05)
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Las tres colas con la misma distribución (α = 0,05)", fontsize=13)
plt.tight_layout()
plt.show()

**Mirá bien:** la curva es la misma en los tres gráficos. Lo único que cambia es **dónde ponemos el área α** (el "5 % de sorpresa" que estamos dispuestos a tolerar), y eso lo decide H₁ **antes de mirar los datos**.

### Ejercicio 1.1
Repetí los tres gráficos con **α = 0,01**. ¿Cómo cambian los valores críticos?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: usá graficar_cola(ax, cola, alpha=0.01) dentro de un bucle como el de arriba

**Tu respuesta:** _(escribí acá)_

### Ejercicio 1.2
Calculá a mano (con `stats.norm.ppf`) los valores críticos del bilateral, izquierda y derecha para α = 0,05 y verificá que coinciden con el gráfico.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: ppf es la inversa de la función de distribución acumulada

## Parte 2 — Test t de una muestra: ¿pesan 3 kg los patitos? 🦆

Medimos 30 patitos. Queremos contrastar si el peso medio poblacional es 3 kg.

- **H₀:** μ = 3 kg
- Probamos las tres alternativas: μ ≠ 3, μ < 3, μ > 3.

Como σ es desconocido, el estadístico sigue una **t de Student con n − 1 gl** bajo H₀.

In [ ]:
pesos = rng.normal(2.8, 0.5, 30)    # muestra de 30 patitos (la media real es 2,8)
mu0, alpha = 3.0, 0.05

print(f"media muestral = {pesos.mean():.3f} kg,  desvío = {pesos.std(ddof=1):.3f} kg")
resultados = {}
for alt in ["two-sided", "less", "greater"]:
    t, p = stats.ttest_1samp(pesos, mu0, alternative=alt)
    resultados[alt] = (t, p)
    decision = "Rechazo H0" if p < alpha else "No rechazo H0"
    print(f"{alt:10s} t={t:.3f}  p={p:.4f}  -> {decision}")

Ahora ubicamos el **t observado** sobre la distribución t(29) en cada cola:

In [ ]:
def graficar_test_t(ax, t_obs, df, cola, alpha=0.05):
    x = np.linspace(-4.5, 4.5, 800)
    y = stats.t.pdf(x, df)
    ax.plot(x, y, color="black", lw=1.5)
    if cola == "two-sided":
        c = stats.t.ppf(1 - alpha / 2, df)
        rech = [x <= -c, x >= c]; crit = [-c, c]
    elif cola == "less":
        c = stats.t.ppf(alpha, df); rech = [x <= c]; crit = [c]
    else:
        c = stats.t.ppf(1 - alpha, df); rech = [x >= c]; crit = [c]
    en_rechazo = any((t_obs <= -abs(cc)) if cc < 0 else (t_obs >= cc) for cc in crit) \
        if cola == "two-sided" else ((t_obs <= crit[0]) if cola == "less" else (t_obs >= crit[0]))
    ax.fill_between(x, y, color="tab:green", alpha=0.2)
    for m in rech:
        ax.fill_between(x, y, where=m, color="tab:red", alpha=0.55)
    for cc in crit:
        ax.axvline(cc, color="tab:red", ls="--", lw=1)
    ax.axvline(t_obs, color="tab:blue", lw=2)
    ax.set_title(f"{cola}\nt obs = {t_obs:.2f}  →  " + ("RECHAZO H₀" if en_rechazo else "no rechazo H₀"), fontsize=10)
    ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, alt in zip(axes, ["two-sided", "less", "greater"]):
    graficar_test_t(ax, resultados[alt][0], df=len(pesos) - 1, cola=alt, alpha=alpha)
plt.tight_layout()
plt.show()

### Ejercicio 2.1
Se pesan 25 patitos de otro criadero:
`[2.9, 3.1, 2.7, 3.3, 2.8, 3.0, 3.2, 2.6, 2.9, 3.4, 3.0, 2.8, 3.1, 2.7, 3.5, 3.2, 2.9, 3.0, 3.3, 2.8, 3.1, 2.9, 3.4, 3.2, 3.0]`

El criador afirma que **pesan más de 3 kg**. Plantéá H₀ y H₁, elegí la cola y decidí con α = 0,05.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: H1: mu > 3 → alternative='greater'

**Tu respuesta:** _(escribí acá)_

## Parte 3 — Errores tipo I y II, potencia y tamaño de muestra

| | **H₀ verdadera** | **H₀ falsa** |
|---|---|---|
| **Rechazo H₀** | ❌ **Error tipo I** (prob. = α) | ✅ Decisión correcta (prob. = 1 − β = **potencia**) |
| **No rechazo H₀** | ✅ Decisión correcta (prob. = 1 − α) | ❌ **Error tipo II** (prob. = β) |

- **α** lo **fija quien analiza** (0,05, 0,01…): es la probabilidad de concluir que μ ≠ 3 cuando en realidad μ = 3.
- **β** **no se fija, se calcula**: depende de α, del tamaño de muestra n, de la variabilidad σ y de **cuán falsa es H₀** (el efecto real).

In [ ]:
mu_real, sigma = 2.8, 0.5
d = abs(mu_real - mu0) / sigma           # tamaño del efecto (Cohen's d)
analisis = TTestPower()

power = analisis.power(effect_size=d, nobs=30, alpha=alpha, alternative="two-sided")
print(f"d = {d:.2f}  |  Potencia = {power:.3f}  |  Beta = {1 - power:.3f}")

n_nec = analisis.solve_power(effect_size=d, power=0.8, alpha=alpha, alternative="two-sided")
print(f"Tamaño de muestra para 80 % de potencia: {np.ceil(n_nec):.0f} patitos")

> ⚠️ Este cálculo es **a priori**: usa el efecto que *suponemos* verdadero (μ = 2,8 y σ = 0,5), no el de la muestra. Además, la potencia **cambia con la cola**: una cola es más potente que dos si el efecto va en la dirección prevista.

### Visualizando α y β
Con σ conocido para simplificar: la curva de H₀ (centrada en μ₀) y la de H₁ (centrada en el valor real). β es el área de H₁ que cae en la región de **no rechazo**.

In [ ]:
n = 30
se = sigma / np.sqrt(n)
x = np.linspace(2.6, 3.2, 600)
f0 = stats.norm.pdf(x, mu0, se)
f1 = stats.norm.pdf(x, mu_real, se)
c_izq = mu0 + stats.norm.ppf(alpha) * se        # cola izquierda (H1: mu < 3)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(x, f0, color="tab:blue", label=r"Distribución bajo $H_0$ ($\mu=3$)")
ax.plot(x, f1, color="tab:orange", label=r"Distribución real ($\mu=2{,}8$)")
ax.fill_between(x, f0, where=x <= c_izq, color="tab:red", alpha=0.5, label=r"$\alpha$ (error tipo I)")
ax.fill_between(x, f1, where=x >= c_izq, color="tab:gray", alpha=0.4, label=r"$\beta$ (error tipo II)")
ax.fill_between(x, f1, where=x <= c_izq, color="tab:green", alpha=0.3, label=r"Potencia $1-\beta$")
ax.axvline(c_izq, color="k", ls="--"); ax.text(c_izq, -0.4, "valor crítico", ha="center", fontsize=9)
ax.set_xlabel("Peso medio muestral (kg)"); ax.set_yticks([])
ax.set_title("α, β y potencia (test de cola izquierda, σ conocido)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### Ejercicio 3.1 — Simulá el error tipo I
Generá 10.000 muestras de 30 patitos **con H₀ verdadera** (μ = 3, σ = 0,5), aplicá el test bilateral con α = 0,05 y calculá la proporción de rechazos. ¿Qué valor esperás?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: bucle con rng.normal(3.0, 0.5, 30) y stats.ttest_1samp

**Tu respuesta:** _(escribí acá)_

### Ejercicio 3.2 — Simulá la potencia
Repetí con **μ real = 2,8** (H₀: μ = 3 sigue siendo falsa). La proporción de rechazos estima la potencia. Compará con el valor teórico de `TTestPower`.

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 3.3
¿Cuántos patitos necesitás para 90 % de potencia? ¿Y si queremos detectar una diferencia menor (μ real = 2,9)?

In [ ]:
# TU CÓDIGO AQUÍ

**Tu respuesta:** _(escribí acá)_

## Parte 4 — Test z de proporciones

Se revisaron 200 patitos y 40 tienen el ala rota. **H₀: p = 0,25** vs **H₁: p < 0,25** (cola izquierda).

En `statsmodels`, las colas se llaman `"smaller"` / `"larger"` / `"two-sided"` (no `"less"`/`"greater"`).

> ⚠️ Para un test de **una** proporción conviene pasar `prop_var=0.25`. Sin ese argumento, `statsmodels` estima el error estándar con la proporción *muestral* en lugar de la de H₀, y el estadístico z cambia levemente.

In [ ]:
z, p = proportions_ztest(count=40, nobs=200, value=0.25, alternative="smaller", prop_var=0.25)
print(f"z = {z:.3f},  p = {p:.4f}  ->", "Rechazo H0" if p < 0.05 else "No rechazo H0")

fig, ax = plt.subplots(figsize=(6, 3.6))
graficar_cola(ax, "less", alpha=0.05, estadistico=z)
ax.set_title("Proporción de alas rotas (cola izquierda)\n$H_0: p=0{,}25$   $H_1: p<0{,}25$")
plt.tight_layout(); plt.show()

### Ejercicio 4.1
En otro criadero, 64 de 200 patitos tienen el ala rota. Contrastá **H₁: p ≠ 0,25** (bilateral) con α = 0,05 y graficá.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: alternative='two-sided' y pasá el z a graficar_cola

## Parte 5 — Otros tests

Estos tests usan estadísticos que **no son simétricos** (χ² y F son siempre ≥ 0), por eso su región de rechazo está **solo en la cola derecha**.

| Test | Pregunta | H₀ | Función |
|---|---|---|---|
| χ² de independencia | ¿dos variables categóricas son independientes? | son independientes | `stats.chi2_contingency` |
| ANOVA de un factor | ¿k grupos tienen la misma media? | μ₁ = μ₂ = … = μₖ | `stats.f_oneway` |
| Mann-Whitney U | ¿dos grupos difieren (sin suponer normalidad)? | misma distribución | `stats.mannwhitneyu` |

### 5.1 Chi-cuadrado de independencia
¿El criadero (A/B) está asociado a tener el ala rota?

In [ ]:
#                 ala rota | ala sana
tabla = np.array([[40, 160],     # criadero A
                  [62, 138]])    # criadero B
chi2, p, gl, esperados = stats.chi2_contingency(tabla)
print(f"chi2 = {chi2:.3f}, gl = {gl}, p = {p:.4f}  ->", "Rechazo H0" if p < 0.05 else "No rechazo H0")
print("Frecuencias esperadas bajo H0:\n", esperados.round(1))

x = np.linspace(0, 15, 600); crit = stats.chi2.ppf(0.95, gl)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(x, stats.chi2.pdf(x, gl), "k")
ax.fill_between(x, stats.chi2.pdf(x, gl), where=x <= crit, color="tab:green", alpha=0.25)
ax.fill_between(x, stats.chi2.pdf(x, gl), where=x >= crit, color="tab:red", alpha=0.55)
ax.axvline(chi2, color="tab:blue", lw=2); ax.set_yticks([])
ax.set_title(f"χ²({gl}): rechazo si χ² > {crit:.2f}   (χ² obs = {chi2:.2f})")
plt.tight_layout(); plt.show()

### 5.2 ANOVA de un factor
¿Tres alimentos distintos producen el mismo peso medio?

In [ ]:
g1 = rng.normal(3.0, 0.4, 20)
g2 = rng.normal(3.1, 0.4, 20)
g3 = rng.normal(3.5, 0.4, 20)
F, p = stats.f_oneway(g1, g2, g3)
print(f"F = {F:.3f}, p = {p:.5f}  ->", "Rechazo H0" if p < 0.05 else "No rechazo H0")

gl1, gl2 = 2, 57; crit = stats.f.ppf(0.95, gl1, gl2)
x = np.linspace(0, 8, 600)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(x, stats.f.pdf(x, gl1, gl2), "k")
ax.fill_between(x, stats.f.pdf(x, gl1, gl2), where=x <= crit, color="tab:green", alpha=0.25)
ax.fill_between(x, stats.f.pdf(x, gl1, gl2), where=x >= crit, color="tab:red", alpha=0.55)
ax.axvline(F, color="tab:blue", lw=2); ax.set_yticks([])
ax.set_title(f"F({gl1},{gl2}): rechazo si F > {crit:.2f}   (F obs = {F:.2f})")
plt.tight_layout(); plt.show()

Un ANOVA significativo solo dice que **alguna** media difiere; para saber cuál hay que hacer comparaciones post-hoc (p. ej. Tukey).

### 5.3 Mann-Whitney U
Dos grupos con pocos datos y distribución claramente asimétrica (no conviene el t).

In [ ]:
a = rng.exponential(1.0, 15)
b = rng.exponential(2.0, 15)
for alt in ["two-sided", "less", "greater"]:
    U, p = stats.mannwhitneyu(a, b, alternative=alt)
    print(f"{alt:10s} U={U:.1f}  p={p:.4f}  ->", "Rechazo H0" if p < 0.05 else "No rechazo H0")

### Ejercicio 5.1 — ANOVA
Con los grupos de abajo (peso de patitos con 3 dietas), probá si las medias son iguales con α = 0,05, graficá la región de rechazo y respondé.

In [ ]:
dieta_a = np.array([2.8, 3.0, 2.9, 3.1, 2.7, 3.0, 2.9, 3.2])
dieta_b = np.array([3.1, 3.3, 3.0, 3.4, 3.2, 3.1, 3.5, 3.3])
dieta_c = np.array([2.9, 3.0, 3.1, 2.8, 3.0, 3.2, 2.9, 3.1])

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: gl1 = k - 1 = 2, gl2 = N - k = 21

**Tu respuesta:** _(escribí acá)_

### Ejercicio 5.2 — Integrador
Elegí **un** test de esta guía y explicá con tus palabras: (a) qué dice H₀ y H₁, (b) en qué cola está la región de rechazo y por qué, (c) qué significaría un error tipo I y uno tipo II en el contexto de los patitos.

**Tu respuesta:** _(escribí acá)_